<a href="https://colab.research.google.com/github/akira-kun/SoulX-FlashHead-Colab/blob/main/Ernie_Image_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #0B0F19 0%, #1e1b4b 100%); border-radius: 16px; padding: 30px; color: white; box-shadow: 0 15px 40px rgba(0,0,0,0.6); border: 1px solid rgba(255,255,255,0.1);">
    <h1 style="color: #ec4899; margin-top: 0; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 800;">✨ ERNIE Image</h1>
    <h3 style="color: #f87171; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">🔴 <b>Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #60a5fa; text-decoration: none;">AI With Chucky</a></b></h3>
    <p style="font-size: 16px; color: #cbd5e1; margin-bottom: 0; margin-top: 15px; line-height: 1.6;">Welcome to the premium interactive environment for ERNIE Image models. This platform dynamically supports both Base and Turbo GGUF architectures.</p>
</div>

### 1. Initialization & Model Configuration
Define your target URLs below to prepare the generative environment. The system securely provisions the backend workspace and utilizes high-speed concurrent connections to fetch the specified foundational models and custom networks.

In [ ]:
# @title ⚙️ Environment Config & 📥 Direct Model Download
UNET_MODEL_URL = "https://huggingface.co/unsloth/ERNIE-Image-Turbo-GGUF/resolve/main/ernie-image-turbo-Q8_0.gguf" # @param {type:"string"}
LORA_MODEL_URL = "" # @param {type:"string"}

import os
from urllib.parse import urlparse
from IPython.display import display, HTML, clear_output

display(HTML("<p style='color:#ec4899; font-family:sans-serif; font-weight:bold;'>🚀 Preparing Generative Environment...</p>"))

# Accelerated Environment Setup
!pip install uv gradio -q > /dev/null 2>&1
!git clone https://github.com/comfyanonymous/ComfyUI.git > /dev/null 2>&1 || true
%cd -q ComfyUI
!uv pip install --system -r requirements.txt > /dev/null 2>&1

# Quantization & Logic Custom Nodes
%cd -q custom_nodes
!git clone https://github.com/city96/ComfyUI-GGUF > /dev/null 2>&1 || true
!uv pip install --system --upgrade gguf > /dev/null 2>&1
!git clone https://github.com/rgthree/rgthree-comfy > /dev/null 2>&1 || true
!git clone https://github.com/pythongosssss/ComfyUI-Custom-Scripts > /dev/null 2>&1 || true
%cd -q ..

clear_output()
display(HTML("<p style='color:#3b82f6; font-family:sans-serif; font-weight:bold;'>📥 Environment ready. Fetching core assets...</p>"))
!apt-get install -y aria2 > /dev/null 2>&1

# Base Model Fulfillment (VAE & CLIP)
!aria2c --console-log-level=error --summary-interval=10 -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/vae/flux2-vae.safetensors -d models/vae -o flux2-vae.safetensors
!aria2c --console-log-level=error --summary-interval=10 -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/ERNIE-Image/resolve/main/text_encoders/ministral-3-3b.safetensors -d models/clip -o ministral-3-3b.safetensors

# Parse and Download UNET (Renamed dynamically to match the expected workflow structure)
UNET_FILE_NAME = "ernie-image-turbo-Q8_0.gguf"
os.environ["UNET_FILE_NAME"] = UNET_FILE_NAME
display(HTML(f"<p style='color:#ec4899; font-family:sans-serif;'>🎯 Retrieving Target Model: <b>{UNET_FILE_NAME}</b></p>"))
!aria2c --console-log-level=error --summary-interval=10 -c -x 16 -s 16 -k 1M {UNET_MODEL_URL} -d models/unet -o {UNET_FILE_NAME}

# Parse and Download LoRA (If Provided)
if LORA_MODEL_URL.strip():
    parsed_lora = urlparse(LORA_MODEL_URL)
    LORA_FILE_NAME = os.path.basename(parsed_lora.path)
    if not LORA_FILE_NAME: LORA_FILE_NAME = "custom_lora.safetensors"
    os.environ["LORA_FILE_NAME"] = LORA_FILE_NAME
    display(HTML(f"<p style='color:#8b5cf6; font-family:sans-serif;'>🪄 Retrieving Target LoRA: <b>{LORA_FILE_NAME}</b></p>"))
    !aria2c --console-log-level=error --summary-interval=10 -c -x 16 -s 16 -k 1M {LORA_MODEL_URL} -d models/loras -o {LORA_FILE_NAME}
else:
    os.environ["LORA_FILE_NAME"] = ""

clear_output()
display(HTML("""
<div style='padding: 16px; background: rgba(16, 185, 129, 0.1); color: #34d399; border-left: 4px solid #10b981; border-radius: 8px; font-family: sans-serif; margin-top: 10px;'>
    <strong>✅ Initialization Complete!</strong> All specified assets have been securely mapped.
</div>
"""))


Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
65ffb5|OK  |   246MiB/s|models/vae/flux2-vae.safetensors

Status Legend:
(OK):download completed.
 *** Download Progress Summary as of Fri Apr 24 17:09:08 2026 *** 
=
[#ba8026 1.7GiB/7.1GiB(24%) CN:16 DL:133MiB ETA:41s]
FILE: models/clip/ministral-3-3b.safetensors
-



### 2. Launch Interface
Deploy the interactive web application. This persistent instance boots the backend engine, routes uncompressed asset pipelines directly to the frontend interface, and secures execution parameters for generation tasks.

In [ ]:
# @title 🎨 Launch Web App
import subprocess
import time
import urllib.request
import urllib.error
import json
import threading
import os
import random
import copy
import gradio as gr
from IPython.display import clear_output, display, HTML

WORKFLOW_URL = "https://github.com/AICHUCKY/Comfyui-Workflows/blob/AICHUCKY-patch-1/Ernie-Iimage.json"

# Auto-convert GitHub blob link to raw content URL for JSON parsing
fetch_url = WORKFLOW_URL
if "github.com" in fetch_url and "/blob/" in fetch_url:
    fetch_url = fetch_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

try:
    req_wf = urllib.request.Request(fetch_url)
    with urllib.request.urlopen(req_wf) as response:
        base_workflow = json.loads(response.read().decode('utf-8'))
except Exception as e:
    print(f"❌ [Error] Could not fetch workflow: {e}")
    base_workflow = {}

# 1. Read Dynamically Downloaded Filenames & Verify Existence
TARGET_UNET = os.environ.get("UNET_FILE_NAME", "ernie-image-turbo-Q8_0.gguf")
TARGET_LORA = os.environ.get("LORA_FILE_NAME", "")

lora_path = f"models/loras/{TARGET_LORA}" if TARGET_LORA else ""
lora_available = bool(TARGET_LORA and os.path.exists(lora_path))

# 2. Start Headless Server & Stream Logs
# Kill any orphaned ComfyUI processes to free port 8188
subprocess.run("fuser -k 8188/tcp", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

print("\n🚀 Initializing Core Engine...")
server = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--use-pytorch-cross-attention"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def stream_logs(process):
    for line in iter(process.stdout.readline, ''):
        print(f"💫 [Engine Log] {line.strip()}")

log_thread = threading.Thread(target=stream_logs, args=(server,))
log_thread.daemon = True
log_thread.start()

# Polling to ensure system is fully loaded before executing tasks
server_ready = False
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        server_ready = True
        break
    except Exception:
        time.sleep(2)

if not server_ready:
    server.terminate()
    raise Exception("Initialization Timeout")

# 3. Web App Generation Logic
def generate_image(prompt, width, height, steps, cfg, seed, enhancer, lora_strength):
    if seed == -1 or seed == 0:
        seed = random.randint(1, 0xffffffffffffffff)

    workflow = copy.deepcopy(base_workflow)

    unet_id, clip_id, sampler_id, clip_encode_id, text_gen_id = None, None, None, None, None

    for node_id, node in workflow.items():
        c_type = node.get("class_type")
        if c_type == "EmptyFlux2LatentImage":
            node["inputs"]["width"] = width
            node["inputs"]["height"] = height
        elif c_type == "PrimitiveStringMultiline":
            node["inputs"]["value"] = prompt
        elif c_type == "KSampler":
            node["inputs"]["seed"] = seed
            node["inputs"]["steps"] = steps
            node["inputs"]["cfg"] = cfg
            sampler_id = node_id
        elif c_type == "UnetLoaderGGUF":
            node["inputs"]["unet_name"] = TARGET_UNET
            unet_id = node_id
        elif c_type == "PrimitiveBoolean":
            node["inputs"]["value"] = enhancer
        elif c_type == "CLIPLoader":
            clip_id = node_id
        elif c_type == "CLIPTextEncode":
            clip_encode_id = node_id
        elif c_type == "TextGenerate":
            text_gen_id = node_id

    if lora_available and lora_strength > 0.0 and all([unet_id, clip_id, sampler_id]):
        workflow["100_LORA"] = {
            "class_type": "LoraLoader",
            "inputs": {
                "lora_name": TARGET_LORA,
                "strength_model": lora_strength,
                "strength_clip": lora_strength,
                "model": [unet_id, 0],
                "clip": [clip_id, 0]
            }
        }
        workflow[sampler_id]["inputs"]["model"] = ["100_LORA", 0]
        if clip_encode_id: workflow[clip_encode_id]["inputs"]["clip"] = ["100_LORA", 1]
        if text_gen_id: workflow[text_gen_id]["inputs"]["clip"] = ["100_LORA", 1]

    req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps({"prompt": workflow}).encode('utf-8'))
    res = json.loads(urllib.request.urlopen(req).read())
    prompt_id = res['prompt_id']

    while True:
        with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as response:
            history = json.loads(response.read())
            if prompt_id in history:
                outputs = history[prompt_id]['outputs']
                for node_id, node_output in outputs.items():
                    if 'images' in node_output:
                        filename = node_output['images'][0]['filename']
                        return os.path.abspath(f"output/{filename}")
        time.sleep(1.5)

# 4. Define Custom CSS for Premium Animated App Interface
css = """
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap');

    @keyframes gradient-bg {
        0% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
        100% { background-position: 0% 50%; }
    }

    @keyframes pulse-glow {
        0% { box-shadow: 0 10px 20px -10px rgba(236, 72, 153, 0.6); }
        50% { box-shadow: 0 15px 30px -5px rgba(236, 72, 153, 0.9); }
        100% { box-shadow: 0 10px 20px -10px rgba(236, 72, 153, 0.6); }
    }

    .gradio-container {
        font-family: 'Inter', sans-serif !important;
        background: linear-gradient(-45deg, #0B0F19, #1e1b4b, #111827, #0f172a) !important;
        background-size: 400% 400% !important;
        animation: gradient-bg 15s ease infinite !important;
        color: #E2E8F0 !important;
        border: none !important;
        max-width: 1400px !important;
        margin: 0 auto !important;
    }

    footer, .gradio-container > .prose { display: none !important; opacity: 0 !important; visibility: hidden !important; }

    .glass-panel {
        background: rgba(30, 41, 59, 0.4) !important;
        backdrop-filter: blur(16px) !important;
        -webkit-backdrop-filter: blur(16px) !important;
        border: 1px solid rgba(255, 255, 255, 0.08) !important;
        border-radius: 16px !important;
        box-shadow: 0 4px 30px rgba(0, 0, 0, 0.2) !important;
        padding: 24px !important;
        overflow: hidden !important;
    }

    .custom-btn {
        background: linear-gradient(135deg, #ec4899 0%, #8b5cf6 100%) !important;
        border: none !important;
        border-radius: 12px !important;
        color: white !important;
        font-weight: 600 !important;
        font-size: 1.1rem !important;
        padding: 14px !important;
        animation: pulse-glow 2.5s infinite ease-in-out !important;
        transition: transform 0.3s ease !important;
    }
    .custom-btn:hover {
        transform: translateY(-2px) !important;
    }

    .header-text {
        text-align: center;
        background: -webkit-linear-gradient(45deg, #ec4899, #8b5cf6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-weight: 800;
        font-size: 2.5rem;
        margin-bottom: 0.5rem;
        letter-spacing: -0.02em;
    }

    input, textarea, .wrap.dark, .slider-container {
        background-color: rgba(15, 23, 42, 0.6) !important;
        border: 1px solid rgba(255, 255, 255, 0.1) !important;
        border-radius: 8px !important;
        color: #F8FAFC !important;
        transition: border-color 0.3s ease !important;
    }
    input:focus, textarea:focus {
        border-color: #ec4899 !important;
    }
"""

with gr.Blocks(theme=gr.themes.Base(), css=css) as ui:
    gr.HTML("""
    <div class='header-text'>ERNIE Image</div>
    <div style='text-align: center; margin-bottom: 2rem;'>
        <h3 style='color: #f87171; font-family: Inter, sans-serif; margin: 0;'>🔴 <b>Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" style="color: #60a5fa; text-decoration: none;">AI With Chucky</a></b></h3>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=4, elem_classes="glass-panel"):
            user_prompt = gr.Textbox(label="Visual Prompt", placeholder="Describe your masterpiece...", lines=4)
            enable_enhancer = gr.Checkbox(label="✨ AI Prompt Enhancer (Auto-expand details)", value=True)

            with gr.Accordion("⚙️ Generation Parameters", open=True):
                with gr.Row():
                    width = gr.Slider(label="Width", minimum=512, maximum=2048, step=64, value=1024)
                    height = gr.Slider(label="Height", minimum=512, maximum=2048, step=64, value=1024)

                with gr.Row():
                    steps = gr.Slider(label="Steps (Turbo = 8, Base = 50)", minimum=4, maximum=100, step=1, value=8)
                    cfg = gr.Slider(label="CFG Scale (Turbo = 1.0, Base = 5.0)", minimum=1.0, maximum=10.0, step=0.5, value=1.0)

                seed = gr.Number(label="Seed (0 for random)", value=0, precision=0)

            with gr.Accordion("🪄 LoRA Settings", open=lora_available):
                lora_status = f"✅ Active LoRA: {TARGET_LORA}" if lora_available else "⚠️ No LoRA Mapped"
                gr.Markdown(f"**Status:** {lora_status}")
                lora_strength = gr.Slider(
                    label="LoRA Network Strength",
                    minimum=0.0, maximum=2.0, step=0.05,
                    value=1.0 if lora_available else 0.0,
                    interactive=lora_available
                )

            gen_btn = gr.Button("🚀 Generate Image", elem_classes="custom-btn")

        with gr.Column(scale=5, elem_classes="glass-panel"):
            output_img = gr.Image(label="Output", show_download_button=True, interactive=False, type="filepath")

    gen_btn.click(
        fn=generate_image,
        inputs=[user_prompt, width, height, steps, cfg, seed, enable_enhancer, lora_strength],
        outputs=[output_img]
    )

clear_output(wait=True)
print("\n====================================================")
print("🟢 WEB APP ONLINE & CELL LOCKED.")
print("Click the public link below to access your Interface.")
print("====================================================\n")
ui.launch(share=True, inline=False, debug=True)